In [1]:
txt = """
I chose the per-class TabGAN/DAE splits to reflect the evaluation metrics and per-class fidelity you provided:

TabGAN outperforms DAE on Macro F1, ROC-AUC, reconstruction error and test loss, so it receives the larger share overall and especially for

Backdoor where the TabGAN class-wise confusion matrix and class statistics best match the original data. I allocate slightly more DAE for the none class (45%) because DAE reproduces some benign feature-statistics and idle/charging state relationships very close to the original (helpful to keep benign variability),

whereas for Backdoor TabGAN preserves the original correlations and discriminative patterns better so it receives the larger share (65%).

For syn-flood I use a 60/40 split favoring TabGAN because both generators approximate syn-flood means and variances well, but TabGAN shows better overall predictive performance and lower reconstruction error on that class, while keeping 40% DAE adds alternative modes and reduces the risk of generator-specific artifacts dominating the node dataset.

| Attack type | TabGAN | DAE |
| ----------- | ------ | --- |
| none        | 55%    | 45% |
| Backdoor    | 65%    | 35% |
| syn-flood   | 60%    | 40% |
"""
print(txt)


I chose the per-class TabGAN/DAE splits to reflect the evaluation metrics and per-class fidelity you provided:

TabGAN outperforms DAE on Macro F1, ROC-AUC, reconstruction error and test loss, so it receives the larger share overall and especially for

Backdoor where the TabGAN class-wise confusion matrix and class statistics best match the original data. I allocate slightly more DAE for the none class (45%) because DAE reproduces some benign feature-statistics and idle/charging state relationships very close to the original (helpful to keep benign variability),

whereas for Backdoor TabGAN preserves the original correlations and discriminative patterns better so it receives the larger share (65%).

For syn-flood I use a 60/40 split favoring TabGAN because both generators approximate syn-flood means and variances well, but TabGAN shows better overall predictive performance and lower reconstruction error on that class, while keeping 40% DAE adds alternative modes and reduces the risk o

In [17]:
# Code Cell 1: Hybrid Synthetic-Original Data Generation Pipeline with Node-Specific Injection

from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

from block_sequences_config import BLOCK_SEQUENCES

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# =============================================================================
# NODE CONFIGURATION (A–H)
# =============================================================================

NODE_TO_PART = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6,
    "G": 7,
    "H": 8,
}

NODES_TO_GENERATE = ["A", "B", "C", "D", "E", "F", "G", "H"]

# =============================================================================
# 1. PATH CONFIGURATION
# =============================================================================

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "dataset_synthetic extension" / "hyperparameter_tuning"
MODELS_DIR = DATA_DIR / "models"
ORIGINAL_DATA_BASE = Path("synthetic_extension")
OUTPUT_DIR = Path("dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_COLUMNS = [
    "shunt_voltage", "bus_voltage_V", "current_mA",
    "power_mW", "State", "Attack"
]
NUMERIC_COLUMNS = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]

BLOCK_SIZE = 1000
TOTAL_BLOCKS = 80

SPLIT_RATIOS = {
    "none": (0.55, 0.45),
    "noneX2": (0.55, 0.45),
    "Backdoor": (0.65, 0.35),
    "syn-flood": (0.60, 0.40),
}

BASE_ATTACK_MAP = {
    "none": "none",
    "noneX2": "none",
    "Backdoor": "Backdoor",
    "syn-flood": "syn-flood",
}

# =============================================================================
# 2. DAE MODEL CLASS
# =============================================================================

class TabularDAE(nn.Module):
    def __init__(
        self,
        input_dim=6,
        hidden_dims=(32, 16),
        latent_dim=18,
        dropout=0.05,
        **kwargs
    ):
        super(TabularDAE, self).__init__()
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.latent_dim = latent_dim
        self.dropout_rate = dropout

        # Encoder: 6 -> 32 -> 16 -> 18
        self.encoder = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[0], self.hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[1], self.latent_dim)
        )

        # Decoder: 18 -> 16 -> 32 -> 6
        self.decoder = nn.Sequential(
            nn.Linear(self.latent_dim, self.hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[1], self.hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[0], self.input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        return self.decoder(latent)

# =============================================================================
# 3. HELPER FUNCTIONS & LOADERS
# =============================================================================

def load_tabgan_model(model_path: Path):
    if not model_path.exists():
        raise FileNotFoundError(f"TabGAN model not found at: {model_path}")
    with open(model_path, "rb") as f:
        model = pickle.load(f)
    print(f"✓ Loaded TabGAN model: {model_path.name}")
    return model


def load_dae_model(model_path: Path) -> TabularDAE:
    if not model_path.exists():
        raise FileNotFoundError(f"DAE model not found at: {model_path}")
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
    
    config = checkpoint.get("model_config", {})
    model = TabularDAE(
        input_dim=config.get("input_dim", 6),
        hidden_dims=config.get("hidden_dims", (32, 16)),
        latent_dim=config.get("latent_dim", 18),
        dropout=config.get("dropout", 0.05)
    )
    
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print(f"✓ Loaded DAE checkpoint: {model_path.name}")
    return model


def load_original_data(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"Original data file not found at: {file_path}")
    df = pd.read_csv(file_path).drop(columns=["time"], errors="ignore")
    df = df[MODEL_COLUMNS]
    return df


def load_and_combine_all_parts():
    """
    Loads and concatenates parts 1 to 8 across all classes to create a single
    unified pooled dataset used for global scalers and baseline sampling references.
    """
    all_class_dfs = {"none": [], "syn-flood": [], "Backdoor": []}

    for part_idx in range(1, 9):
        for atk in ["none", "syn-flood", "Backdoor"]:
            file_p = ORIGINAL_DATA_BASE / f"{atk}_part_{part_idx}.csv"
            if file_p.exists():
                df = pd.read_csv(file_p).drop(columns=["time"], errors="ignore")[MODEL_COLUMNS]
                all_class_dfs[atk].append(df)

    combined_dict = {
        atk: pd.concat(dfs, ignore_index=True) for atk, dfs in all_class_dfs.items() if len(dfs) > 0
    }
    return combined_dict


def prepare_global_preprocessors(combined_data_dict):
    scalers = {}
    feature_cols_map = {}

    for attack_class, df in combined_data_dict.items():
        df_encoded = pd.get_dummies(df, columns=["State"], drop_first=False)
        
        for state_col in ["State_charging", "State_idle"]:
            if state_col not in df_encoded.columns:
                df_encoded[state_col] = 0

        feature_cols = NUMERIC_COLUMNS + ["State_charging", "State_idle"]

        scaler = StandardScaler()
        scaler.fit(df_encoded[feature_cols])

        scalers[attack_class] = scaler
        feature_cols_map[attack_class] = feature_cols
        print(f"✓ Global StandardScaler fitted for class '{attack_class}' ({len(df)} total rows across all parts)")

    return scalers, feature_cols_map


def calculate_row_counts(block_size: int, tabgan_ratio: float, dae_ratio: float):
    tabgan_rows = round(block_size * tabgan_ratio)
    dae_rows = block_size - tabgan_rows
    return tabgan_rows, dae_rows


def sample_from_tabgan(model, n_samples: int, block_type: str, pooled_all_parts_data: pd.DataFrame) -> pd.DataFrame:
    try:
        generator = model.get_object_generator() if hasattr(model, "get_object_generator") else model
        
        if hasattr(generator, "synthesizer") and hasattr(generator.synthesizer, "sample"):
            df = generator.synthesizer.sample(n_samples)
        elif hasattr(generator, "_model") and hasattr(generator._model, "sample"):
            df = generator._model.sample(n_samples)
        elif hasattr(model, "sample"):
            df = model.sample(n_samples)
        else:
            df = pooled_all_parts_data.sample(n=n_samples, replace=True).copy().reset_index(drop=True)

        for col in MODEL_COLUMNS:
            if col not in df.columns:
                if col == "Attack":
                    df["Attack"] = block_type
                elif col == "State":
                    df["State"] = np.random.choice(["idle", "charging"], size=len(df))
                else:
                    df[col] = pooled_all_parts_data[col].sample(n=len(df), replace=True).values

        df = df[MODEL_COLUMNS].copy()
        if len(df) > n_samples:
            df = df.iloc[:n_samples].reset_index(drop=True)
        elif len(df) < n_samples:
            extra = df.sample(n=n_samples - len(df), replace=True)
            df = pd.concat([df, extra], ignore_index=True)

        for col in NUMERIC_COLUMNS:
            df[col] = np.clip(df[col], a_min=0, a_max=None)

        df["Attack"] = block_type
        df["data_source"] = "TabGAN"
        return df

    except Exception as e:
        print(f"  ✗ Sampling from TabGAN failed, falling back to pooled data sample: {e}")
        df = pooled_all_parts_data.sample(n=n_samples, replace=True).copy().reset_index(drop=True)
        df["Attack"] = block_type
        df["data_source"] = "TabGAN"
        return df


def sample_from_dae(
    dae_model: TabularDAE, 
    scaler: StandardScaler, 
    feature_cols: list, 
    n_samples: int, 
    block_type: str, 
    pooled_all_parts_data: pd.DataFrame
) -> pd.DataFrame:
    try:
        raw_samples = pooled_all_parts_data.sample(n=n_samples, replace=True).reset_index(drop=True)
        df_encoded = pd.get_dummies(raw_samples, columns=["State"], drop_first=False)
        
        for col in feature_cols:
            if col not in df_encoded.columns:
                df_encoded[col] = 0

        x_scaled = scaler.transform(df_encoded[feature_cols])
        x_tensor = torch.tensor(x_scaled, dtype=torch.float32)

        with torch.no_grad():
            latent = dae_model.encoder(x_tensor)
            latent_perturbed = latent + torch.randn_like(latent) * 0.05
            reconstructed_scaled = dae_model.decoder(latent_perturbed).detach().cpu().numpy()

        reconstructed_physical = scaler.inverse_transform(reconstructed_scaled)
        df_reconstructed = pd.DataFrame(reconstructed_physical, columns=feature_cols)

        for col in NUMERIC_COLUMNS:
            df_reconstructed[col] = np.clip(df_reconstructed[col], a_min=0, a_max=None)

        state_cols = ["State_charging", "State_idle"]
        state_idx = np.argmax(df_reconstructed[state_cols].values, axis=1)
        df_reconstructed["State"] = [state_cols[i].replace("State_", "") for i in state_idx]

        df_reconstructed["Attack"] = block_type
        df_reconstructed["data_source"] = "DAE"

        return df_reconstructed[MODEL_COLUMNS + ["data_source"]]

    except Exception as e:
        print(f"  ✗ Error sampling from DAE ({block_type}): {e}")
        raise


def replace_with_original(block_df: pd.DataFrame, original_slice: pd.DataFrame) -> pd.DataFrame:
    """
    Injects node-specific original data (part_N) into specific rows of the synthetic block.
    """
    block = block_df.copy()
    n_orig = min(len(original_slice), len(block))
    if n_orig == 0:
        return block

    replace_idx = np.random.choice(block.index, size=n_orig, replace=False)
    orig_rows = original_slice[MODEL_COLUMNS].iloc[:n_orig].copy()
    orig_rows["data_source"] = "Original"

    block.loc[replace_idx, MODEL_COLUMNS + ["data_source"]] = orig_rows.values
    return block

# =============================================================================
# 4. MAIN PIPELINE (Hybrid with Original Data Injection)
# =============================================================================

def generate_synthetic_dataset(
    node_id: str,
    tabgan_models: dict,
    dae_models: dict,
    global_scalers: dict,
    feature_cols_map: dict,
    pooled_all_parts_data: dict
) -> pd.DataFrame:
    if node_id not in BLOCK_SEQUENCES:
        raise ValueError(f"Unknown node_id: {node_id}")

    BLOCK_SEQUENCE = BLOCK_SEQUENCES[node_id][:TOTAL_BLOCKS]
    part_index = NODE_TO_PART[node_id]

    NODE_ORIGINAL_FILES = {
        "none": ORIGINAL_DATA_BASE / f"none_part_{part_index}.csv",
        "syn-flood": ORIGINAL_DATA_BASE / f"syn-flood_part_{part_index}.csv",
        "Backdoor": ORIGINAL_DATA_BASE / f"Backdoor_part_{part_index}.csv",
    }
    node_original_data = {atk: load_original_data(p) for atk, p in NODE_ORIGINAL_FILES.items()}

    OUTPUT_FILE = OUTPUT_DIR / f"Node_{node_id}_final_synthetic_dataset_with_source.csv"

    print("=" * 80)
    print(f"HYBRID PIPELINE WITH ORIGINAL INJECTION (Node {node_id}, Blocks: {len(BLOCK_SEQUENCE)}, Part: {part_index})")
    print("=" * 80)

    supported_blocks = list(SPLIT_RATIOS.keys())
    attack_block_counts = {atk: BLOCK_SEQUENCE.count(atk) for atk in supported_blocks}
    attack_block_seen = {atk: 0 for atk in supported_blocks}
    original_cursors = {atk: 0 for atk in supported_blocks}

    blocks = []
    cols_with_source = MODEL_COLUMNS + ["data_source"]

    for block_type in tqdm(BLOCK_SEQUENCE, desc=f"Generating blocks (Node {node_id})"):
        base_type = BASE_ATTACK_MAP.get(block_type, block_type)
        attack_block_seen[block_type] += 1
        tab_rows, dae_rows = calculate_row_counts(BLOCK_SIZE, *SPLIT_RATIOS[block_type])

        # 1. Sample from TabGAN
        tab_df = sample_from_tabgan(tabgan_models[base_type], tab_rows, base_type, pooled_all_parts_data[base_type])

        # 2. Sample from DAE
        dae_df = sample_from_dae(
            dae_models[base_type],
            global_scalers[base_type],
            feature_cols_map[base_type],
            dae_rows,
            base_type,
            pooled_all_parts_data[base_type]
        )

        block_df = pd.concat([tab_df[cols_with_source], dae_df[cols_with_source]], ignore_index=True)

        if len(block_df) > BLOCK_SIZE:
            block_df = block_df.iloc[:BLOCK_SIZE].reset_index(drop=True)
        elif len(block_df) < BLOCK_SIZE:
            extra = block_df.sample(n=BLOCK_SIZE - len(block_df), replace=True)
            block_df = pd.concat([block_df, extra], ignore_index=True)

        # 3. Inject node-specific original data slice
        total_rows = len(node_original_data[base_type])
        n_blocks = attack_block_counts[block_type]
        rows_per_block = total_rows // n_blocks if n_blocks > 0 else 0
        
        start_idx = original_cursors[block_type]
        end_idx = total_rows if (n_blocks > 0 and attack_block_seen[block_type] == n_blocks) else min(start_idx + rows_per_block, total_rows)

        orig_slice = node_original_data[base_type].iloc[start_idx:end_idx].reset_index(drop=True)
        original_cursors[block_type] = end_idx

        block_df = replace_with_original(block_df, orig_slice)

        # Apply multiplier 1.3 for noneX2
        if block_type == "noneX2":
            block_df[NUMERIC_COLUMNS] = block_df[NUMERIC_COLUMNS] * 1.3

        block_df["Attack"] = block_type
        block_df = block_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        blocks.append(block_df)

    final_df = pd.concat(blocks, ignore_index=True)

    print("\n" + "=" * 80)
    print(f"NEGATIVE VALUE CHECK REPORT (Node {node_id})")
    print("=" * 80)
    for col in NUMERIC_COLUMNS:
        neg_count = (final_df[col] < 0).sum()
        print(f"Feature '{col}': {neg_count} negative values")

    print(f"\nFinal dataset sample (Head, Node {node_id}):")
    print(final_df.head(10))

    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✓ Saved {len(final_df)} rows to: {OUTPUT_FILE}")
    return final_df

# =============================================================================
# 5. EXECUTION
# =============================================================================

if __name__ == "__main__":
    TABGAN_MODELS = {
        "none": MODELS_DIR / "tabgan_generator_none.pkl",
        "syn-flood": MODELS_DIR / "tabgan_generator_syn-flood.pkl",
        "Backdoor": MODELS_DIR / "tabgan_generator_Backdoor.pkl",
    }

    DAE_MODELS = {
        "none": MODELS_DIR / "dae_none.pt",
        "syn-flood": MODELS_DIR / "dae_syn-flood.pt",
        "Backdoor": MODELS_DIR / "dae_Backdoor.pt",
    }

    print("\nSTEP 1: Loading pre-trained models")
    print("-" * 80)
    tabgan_models = {atk: load_tabgan_model(p) for atk, p in TABGAN_MODELS.items()}
    dae_models = {atk: load_dae_model(p) for atk, p in DAE_MODELS.items()}

    print("\nSTEP 2: Pooling all parts (1–8) for global scaling & shared reference")
    print("-" * 80)
    pooled_all_parts_data = load_and_combine_all_parts()
    global_scalers, feature_cols_map = prepare_global_preprocessors(pooled_all_parts_data)

    generated_datasets = {}
    for node_id in NODES_TO_GENERATE:
        print("\n" + "=" * 80)
        print(f"GENERATING NODE {node_id}")
        print("=" * 80)
        df_node = generate_synthetic_dataset(
            node_id=node_id,
            tabgan_models=tabgan_models,
            dae_models=dae_models,
            global_scalers=global_scalers,
            feature_cols_map=feature_cols_map,
            pooled_all_parts_data=pooled_all_parts_data
        )
        generated_datasets[node_id] = df_node


STEP 1: Loading pre-trained models
--------------------------------------------------------------------------------
✓ Loaded TabGAN model: tabgan_generator_none.pkl
✓ Loaded TabGAN model: tabgan_generator_syn-flood.pkl
✓ Loaded TabGAN model: tabgan_generator_Backdoor.pkl
✓ Loaded DAE checkpoint: dae_none.pt
✓ Loaded DAE checkpoint: dae_syn-flood.pt
✓ Loaded DAE checkpoint: dae_Backdoor.pt

STEP 2: Pooling all parts (1–8) for global scaling & shared reference
--------------------------------------------------------------------------------
✓ Global StandardScaler fitted for class 'none' (14363 total rows across all parts)
✓ Global StandardScaler fitted for class 'syn-flood' (13517 total rows across all parts)
✓ Global StandardScaler fitted for class 'Backdoor' (21137 total rows across all parts)

GENERATING NODE A
HYBRID PIPELINE WITH ORIGINAL INJECTION (Node A, Blocks: 80, Part: 1)


Generating blocks (Node A): 100%|██████████████| 80/80 [00:05<00:00, 13.68it/s]



NEGATIVE VALUE CHECK REPORT (Node A)
Feature 'shunt_voltage': 0 negative values
Feature 'bus_voltage_V': 0 negative values
Feature 'current_mA': 0 negative values
Feature 'power_mW': 0 negative values

Final dataset sample (Head, Node A):
   shunt_voltage  bus_voltage_V  current_mA     power_mW     State Attack  \
0     497.000000       5.193000  571.000000  2940.000000  charging   none   
1     528.193787       5.196590  507.595215  2800.493164  charging   none   
2     565.733826       5.196071  530.334045  2717.637939      idle   none   
3     640.951782       5.189672  575.172119  2985.810547      idle   none   
4     429.000000       5.201000  432.000000  2260.000000      idle   none   
5     443.998535       5.200850  428.005219  2264.382812      idle   none   
6     799.173950       5.185634  606.285889  3151.735352  charging   none   
7     661.000000       5.193000  660.000000  4040.000000  charging   none   
8     437.981323       5.200446  435.882172  2295.041016      idle 

Generating blocks (Node B): 100%|██████████████| 80/80 [00:05<00:00, 15.20it/s]



NEGATIVE VALUE CHECK REPORT (Node B)
Feature 'shunt_voltage': 0 negative values
Feature 'bus_voltage_V': 0 negative values
Feature 'current_mA': 0 negative values
Feature 'power_mW': 0 negative values

Final dataset sample (Head, Node B):
   shunt_voltage  bus_voltage_V  current_mA     power_mW     State Attack  \
0     499.000000       5.197000  498.000000  2560.000000      idle   none   
1     645.979675       5.189657  649.125916  3288.664062  charging   none   
2     544.013733       5.192699  511.344971  2817.307617  charging   none   
3     437.709778       5.201535  440.743134  2245.698975      idle   none   
4     586.000000       5.185000  516.000000  2720.000000  charging   none   
5     517.299194       5.198794  590.065247  2957.311523  charging   none   
6     544.471436       5.194037  549.238342  2954.434570      idle   none   
7     548.000000       5.189000  612.000000  2820.000000  charging   none   
8     543.077087       5.196161  549.522461  2892.123291      idle 

...


NEGATIVE VALUE CHECK REPORT (Node F)
Feature 'shunt_voltage': 0 negative values
Feature 'bus_voltage_V': 0 negative values
Feature 'current_mA': 0 negative values
Feature 'power_mW': 0 negative values

Final dataset sample (Head, Node F):
   shunt_voltage  bus_voltage_V  current_mA     power_mW     State Attack  \
0     520.000000       5.193000  699.000000  3300.000000  charging   none   
1     540.707703       5.198719  517.553467  2660.884521  charging   none   
2     510.844360       5.196861  525.221985  2642.472412      idle   none   
3     760.729858       5.194051  533.137634  2646.973389  charging   none   
4     511.000000       5.189000  649.000000  3120.000000  charging   none   
5     512.973999       5.192687  565.369141  3666.738037  charging   none   
6     439.210693       5.200573  438.421875  2277.550781      idle   none   
7     430.000000       5.201000  435.000000  2200.000000      idle   none   
8     478.667480       5.200089  444.474060  2312.130127      idle 

Generating blocks (Node G): 100%|██████████████| 80/80 [00:05<00:00, 14.53it/s]



NEGATIVE VALUE CHECK REPORT (Node G)
Feature 'shunt_voltage': 0 negative values
Feature 'bus_voltage_V': 0 negative values
Feature 'current_mA': 0 negative values
Feature 'power_mW': 0 negative values

Final dataset sample (Head, Node G):
   shunt_voltage  bus_voltage_V  current_mA     power_mW     State Attack  \
0     604.000000       5.177000  557.000000  2880.000000  charging   none   
1     433.000000       5.201000  436.000000  2260.000000      idle   none   
2     433.013672       5.201050  444.837830  2409.206299      idle   none   
3     521.362549       5.196584  570.363586  2997.997070  charging   none   
4     436.000000       5.205000  434.000000  2620.000000      idle   none   
5     468.470856       5.200541  451.875824  2344.318115      idle   none   
6     511.296783       5.197248  517.787781  2685.606934      idle   none   
7     755.000000       5.181000  638.000000  3280.000000  charging   none   
8     531.960144       5.196838  595.796021  3223.191895  charging 

Generating blocks (Node H): 100%|██████████████| 80/80 [00:05<00:00, 14.62it/s]



NEGATIVE VALUE CHECK REPORT (Node H)
Feature 'shunt_voltage': 0 negative values
Feature 'bus_voltage_V': 0 negative values
Feature 'current_mA': 0 negative values
Feature 'power_mW': 0 negative values

Final dataset sample (Head, Node H):
   shunt_voltage  bus_voltage_V  current_mA     power_mW     State Attack  \
0     525.000000       5.197000  540.000000  3140.000000  charging   none   
1     532.874451       5.196621  538.605835  2826.231689  charging   none   
2     504.379852       5.192688  552.083618  3069.481934  charging   none   
3     450.758423       5.199747  454.903015  2361.222412      idle   none   
4     447.000000       5.205000  447.000000  2360.000000      idle   none   
5     548.348145       5.196428  728.097168  3698.406494  charging   none   
6     506.577698       5.196993  524.213013  2699.116211      idle   none   
7     595.000000       5.185000  553.000000  2720.000000  charging   none   
8     619.271851       5.185552  824.689209  3776.734131  charging 

In [5]:
# Verification Code Cell: Global Aggregate Data Source Breakdown per Attack Type (All Nodes Combined)

import pandas as pd
from pathlib import Path

nodes = ["A", "B", "C", "D", "E", "F", "G", "H"]
output_dir = Path("dataset")

all_dfs = []
for node_id in nodes:
    file_path = output_dir / f"Node_{node_id}_final_synthetic_dataset_with_source.csv"
    if file_path.exists():
        df = pd.read_csv(file_path)
        all_dfs.append(df)

if all_dfs:
    # Concatenate all node datasets into a single global DataFrame
    global_df = pd.concat(all_dfs, ignore_index=True)
    
    # Calculate percentage distribution of data sources across all nodes for each attack type
    global_breakdown = pd.crosstab(global_df['Attack'], global_df['data_source'], normalize='index') * 100
    
    # Ensure standard column order if present
    cols = [col for col in ['TabGAN', 'DAE', 'Original'] if col in global_breakdown.columns]
    global_breakdown = global_breakdown[cols]
    
    print("\nGlobal Aggregate - Data Source Breakdown per Attack Type (% across all Nodes):")
    print("=" * 70)
    print(global_breakdown.round(2).to_string())
    print("=" * 70)
else:
    print("No datasets found. Please ensure the generation script has been run.")


Global Aggregate - Data Source Breakdown per Attack Type (% across all Nodes):
data_source  TabGAN    DAE  Original
Attack                              
Backdoor      55.46  29.86     14.68
none          52.40  42.86      4.74
noneX2        35.39  28.70     35.91
syn-flood     54.87  36.58      8.56


In [10]:
txt = "Für die Evaluation wurde eine hybride Datengenerierungspipeline entwickelt, die auf einem global gepoolten Referenzdatensatz basiert und synthetische Daten mit knotenspezifischen Originalmessreihen (Parts 1–8) verknüpft. Die klassenspezifischen Syntheseverhältnisse zwischen TabGAN und DAE berücksichtigen die empirische Modellgüte: Für Backdoor dominiert TabGAN mit 65 % (35 % DAE) aufgrund optimaler Diskriminierungsmuster, die Gutklasse (none) nutzt einen DAE-Anteil von 45 % (55 % TabGAN) zur originalgetreuen Erfassung von Sensor- und Zustandsvarianz, während bei syn-flood eine 60/40-Aufteilung Artefakte minimiert. Über acht Knoten (A–H) wurden so sequenziell 80 Blöcke zur präzisen Analyse des Robustheits-Sensitivitäts-Konflikts erzeugt.-"
print(txt)

Für die Evaluation wurde eine hybride Datengenerierungspipeline entwickelt, die auf einem global gepoolten Referenzdatensatz basiert und synthetische Daten mit knotenspezifischen Originalmessreihen (Parts 1–8) verknüpft. Die klassenspezifischen Syntheseverhältnisse zwischen TabGAN und DAE berücksichtigen die empirische Modellgüte: Für Backdoor dominiert TabGAN mit 65 % (35 % DAE) aufgrund optimaler Diskriminierungsmuster, die Gutklasse (none) nutzt einen DAE-Anteil von 45 % (55 % TabGAN) zur originalgetreuen Erfassung von Sensor- und Zustandsvarianz, während bei syn-flood eine 60/40-Aufteilung Artefakte minimiert. Über acht Knoten (A–H) wurden so sequenziell 80 Blöcke zur präzisen Analyse des Robustheits-Sensitivitäts-Konflikts erzeugt.-
